Index | [Python](Chaines-11-Groupe-Lorentz-Signature-Un-Contre-Trois-Python.ipynb)

# Chaînes 11 — Le groupe de Lorentz et la signature un contre trois

Chapitre 50 du compendium *Chain Theory* (Wissner-Gross, 2026, preprint non relu), sections 54-55. Deux résultats : le **Théorème 54.1** (conditionnel) — le groupe de Lorentz comme isométries de la scission hermitienne — et le **Théorème 55.1**, premier résultat de l'arc au statut de théorème établi : la signature un contre trois vit au choix de l'unité et le transport la porte le long de chaque chaîne sans bouger.

Le patron de la série tient : **Mesuré** ce que nos `Fract` rejouent, **Cité** toute la chaîne dérivative du source, **Ouvert** ce qu'il ne prouve pas.

## Le cadre : l'ordre rejoint les rotations

« The rotations fixed, the next chapter joins them to the order coordinate in the Lorentz group » (p. 143). Le chapitre 49 avait dérivé les rotations des directions ; celui-ci y adjoint les boosts — les générateurs qui **mélangent** l'ordre et la direction.

Les statuts diffèrent et le carnet les garde visibles : le 54.1 est **conditionnel** (« conditional on the axiom text and on the results of the chapters named as its premises »), le 55.1 porte la mention « **Standing: theorem** » — la première de la Partie IV sur ce tronçon.

## §54 — Théorème 54.1 (conditionnel) : le groupe de Lorentz comme isométries de la scission hermitienne

« The isometry algebra of the Hermitian split at the qutrit-qubit link [...] is six-dimensional, spanned by the ad matrices of the directions, which fix the unit, and by three boosts generated from the hyperbolic generator by the rotation brackets; at every structure constant it equals the Lorentz algebra of the literature by name, with the Killing form of signature three and three, and the six lift to the qubit end as three anti-Hermitian and three Hermitian generators » (p. 144).

**Notre modèle, déclaré nôtre** : l'espace $\mathbb{R}^4$ de coordonnées $(t, x, y, z)$ muni de $\eta = \mathrm{diag}(4, -1, -1, -1)$ — **la Gram de propagation de l'éq. 81 elle-même** —, trois rotations de bloc spatial (les $\mathrm{ad}$ matrices des directions de la construction, citées), trois boosts dont le bloc $2\times2$ est le générateur littéral imprimé de l'éq. 82. Tout ce qui suit se mesure en `Fract` exacte sur ce modèle ; l'égalité « by name » à l'algèbre de la construction reste citée.

In [2]:
// Arithmetique exacte sur Q : matrices 4x4 en Vec<Vec<Fract>>.
use std::cmp::Ordering;
use std::fmt;
use std::ops::{Add, Sub, Mul, Div, Neg};

#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: u128, mut b: u128) -> u128 {
    while b != 0 { let t = a % b; a = b; b = t; } a
}

impl Fract {
    fn new(n: i128, d: i128) -> Self {
        assert!(d != 0);
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = gcd(n.unsigned_abs(), d as u128) as i128;
        Self { num: n / g, den: d / g }
    }
    fn int(n: i128) -> Self { Self::new(n, 1) }
}

impl Add for Fract { type Output=Self; fn add(self,r:Self)->Self { Self::new(self.num*r.den+r.num*self.den,self.den*r.den) } }
impl Sub for Fract { type Output=Self; fn sub(self,r:Self)->Self { Self::new(self.num*r.den-r.num*self.den,self.den*r.den) } }
impl Mul for Fract { type Output=Self; fn mul(self,r:Self)->Self { Self::new(self.num*r.num,self.den*r.den) } }
impl Div for Fract { type Output=Self; fn div(self,r:Self)->Self { Self::new(self.num*r.den,self.den*r.num) } }
impl Neg for Fract { type Output=Self; fn neg(self)->Self { Self::new(-self.num,self.den) } }
impl PartialOrd for Fract { fn partial_cmp(&self,r:&Self)->Option<Ordering> { (self.num*r.den).partial_cmp(&(r.num*self.den)) } }
impl fmt::Display for Fract {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

fn mul(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let n = A.len(); let m = B[0].len(); let k = B.len();
    let mut R = vec![vec![Fract::int(0); m]; n];
    for i in 0..n { for j in 0..m {
        let mut s = Fract::int(0);
        for t in 0..k { s = s + A[i][t] * B[t][j]; }
        R[i][j] = s;
    }}
    R
}
fn add(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = A[i][j] + B[i][j]; } }
    R
}
fn sub(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = A[i][j] - B[i][j]; } }
    R
}
fn br(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    sub(&mul(A, B), &mul(B, A))
}
fn eq(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> bool {
    A.iter().zip(B.iter()).all(|(ra, rb)| ra.iter().zip(rb.iter()).all(|(a, b)| a == b))
}
fn transp(A: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let n = A.len();
    let mut R = vec![vec![Fract::int(0); n]; n];
    for i in 0..n { for j in 0..n { R[i][j] = A[j][i]; } }
    R
}
fn neg(A: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = -A[i][j]; } }
    R
}
fn zeros(n: usize) -> Vec<Vec<Fract>> {
    vec![vec![Fract::int(0); n]; n]
}
fn E(i: usize, j: usize) -> Vec<Vec<Fract>> {
    let mut M = zeros(4); M[i][j] = Fract::int(1); M
}

// L'appariement : eta = diag(4, -1, -1, -1) -- la Gram de propagation de l'eq. 81.
let eta = vec![
    vec![Fract::int(4), Fract::int(0), Fract::int(0), Fract::int(0)],
    vec![Fract::int(0), Fract::int(-1), Fract::int(0), Fract::int(0)],
    vec![Fract::int(0), Fract::int(0), Fract::int(-1), Fract::int(0)],
    vec![Fract::int(0), Fract::int(0), Fract::int(0), Fract::int(-1)],
];

// Trois rotations (blocs spatiaux, (Ji)_jk = -eps_ijk) et trois boosts (plans (1/2, 2)).
let J1 = sub(&E(3, 2), &E(2, 3));
let J2 = sub(&E(1, 3), &E(3, 1));
let J3 = sub(&E(2, 1), &E(1, 2));
fn boost(i: usize) -> Vec<Vec<Fract>> {
    let mut K = zeros(4);
    K[0][i] = Fract::new(1, 2);
    K[i][0] = Fract::int(2);
    K
}
let K1 = boost(1);
let K2 = boost(2);
let K3 = boost(3);
let base = vec![J1, J2, J3, K1, K2, K3];
let noms: Vec<&str> = vec!["J1", "J2", "J3", "K1", "K2", "K3"];

// Condition d'algebre : X^T eta + eta X = 0 pour les six.
fn residu(X: &Vec<Vec<Fract>>, P: &Vec<Vec<Fract>>) -> bool {
    let R = add(&mul(&transp(X), P), &mul(P, X));
    R.iter().all(|row| row.iter().all(|&v| v == Fract::int(0)))
}
let mut residus: Vec<(&str, bool)> = Vec::new();
for i in 0..6 {
    residus.push((noms[i], residu(&base[i], &eta)));
}

fn py_bool(b: bool) -> String {
    if b { "True".to_string() } else { "False".to_string() }
}
fn py_str_bool_dict(v: &Vec<(&str, bool)>) -> String {
    let parts: Vec<String> = v.iter().map(|&(k, b)| format!("'{}': {}", k, py_bool(b))).collect();
    format!("{{{}}}", parts.join(", "))
}

println!("X^T eta + eta X = 0 pour les six generateurs -> {} -> {}", py_str_bool_dict(&residus), py_bool(residus.iter().all(|&(_, b)| b)));

X^T eta + eta X = 0 pour les six generateurs -> {'J1': True, 'J2': True, 'J3': True, 'K1': True, 'K2': True, 'K3': True} -> True


### Lecture du résultat (isométries)

**Mesuré** : les six générateurs — trois rotations qui fixent l'axe du temps (les « ad matrices of the directions » de la construction, citées) et trois boosts — satisfont la condition d'algèbre d'isométrie de $\eta$. Le bloc $2\times2$ de chaque boost, $\begin{psmallmatrix} 0 & 1/2 \\ 2 & 0 \end{psmallmatrix}$, est le générateur imprimé de l'éq. 82 : notre modèle et la construction partagent les mêmes matrices de plan.

In [3]:
// Les crochets de l'eq. 80 : [J1,J2] = J3, [Ji,Kj] = eps_ijk Kk, [K1,K2] = -J3.
use std::cmp::Ordering;
use std::fmt;
use std::ops::{Add, Sub, Mul, Div, Neg};

#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: u128, mut b: u128) -> u128 {
    while b != 0 { let t = a % b; a = b; b = t; } a
}

impl Fract {
    fn new(n: i128, d: i128) -> Self {
        assert!(d != 0);
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = gcd(n.unsigned_abs(), d as u128) as i128;
        Self { num: n / g, den: d / g }
    }
    fn int(n: i128) -> Self { Self::new(n, 1) }
}

impl Add for Fract { type Output=Self; fn add(self,r:Self)->Self { Self::new(self.num*r.den+r.num*self.den,self.den*r.den) } }
impl Sub for Fract { type Output=Self; fn sub(self,r:Self)->Self { Self::new(self.num*r.den-r.num*self.den,self.den*r.den) } }
impl Mul for Fract { type Output=Self; fn mul(self,r:Self)->Self { Self::new(self.num*r.num,self.den*r.den) } }
impl Div for Fract { type Output=Self; fn div(self,r:Self)->Self { Self::new(self.num*r.den,self.den*r.num) } }
impl Neg for Fract { type Output=Self; fn neg(self)->Self { Self::new(-self.num,self.den) } }
impl PartialOrd for Fract { fn partial_cmp(&self,r:&Self)->Option<Ordering> { (self.num*r.den).partial_cmp(&(r.num*self.den)) } }
impl fmt::Display for Fract {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

fn mul(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let n = A.len(); let m = B[0].len(); let k = B.len();
    let mut R = vec![vec![Fract::int(0); m]; n];
    for i in 0..n { for j in 0..m {
        let mut s = Fract::int(0);
        for t in 0..k { s = s + A[i][t] * B[t][j]; }
        R[i][j] = s;
    }}
    R
}
fn add(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = A[i][j] + B[i][j]; } }
    R
}
fn sub(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = A[i][j] - B[i][j]; } }
    R
}
fn br(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    sub(&mul(A, B), &mul(B, A))
}
fn eq(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> bool {
    A.iter().zip(B.iter()).all(|(ra, rb)| ra.iter().zip(rb.iter()).all(|(a, b)| a == b))
}
fn transp(A: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let n = A.len();
    let mut R = vec![vec![Fract::int(0); n]; n];
    for i in 0..n { for j in 0..n { R[i][j] = A[j][i]; } }
    R
}
fn neg(A: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = -A[i][j]; } }
    R
}
fn zeros(n: usize) -> Vec<Vec<Fract>> {
    vec![vec![Fract::int(0); n]; n]
}
fn E(i: usize, j: usize) -> Vec<Vec<Fract>> {
    let mut M = zeros(4); M[i][j] = Fract::int(1); M
}

let J1 = sub(&E(3, 2), &E(2, 3));
let J2 = sub(&E(1, 3), &E(3, 1));
let J3 = sub(&E(2, 1), &E(1, 2));
fn boost(i: usize) -> Vec<Vec<Fract>> {
    let mut K = zeros(4);
    K[0][i] = Fract::new(1, 2);
    K[i][0] = Fract::int(2);
    K
}
let K1 = boost(1);
let K2 = boost(2);
let K3 = boost(3);
let base = vec![J1.clone(), J2.clone(), J3.clone(), K1.clone(), K2.clone(), K3.clone()];

fn eps(i: usize, j: usize, k: usize) -> i64 {
    if i == j || j == k || i == k { return 0; }
    if (i, j, k) == (1, 2, 3) || (i, j, k) == (2, 3, 1) || (i, j, k) == (3, 1, 2) { 1 } else { -1 }
}

let rot_rot = vec![
    eq(&br(&J1, &J2), &J3),
    eq(&br(&J2, &J3), &J1),
    eq(&br(&J3, &J1), &J2),
];
fn py_bools(v: &Vec<bool>) -> String {
    let parts: Vec<String> = v.iter().map(|&b| if b { "True".to_string() } else { "False".to_string() }).collect();
    format!("[{}]", parts.join(", "))
}
fn py_bool(b: bool) -> String {
    if b { "True".to_string() } else { "False".to_string() }
}
println!("[J1,J2]=J3, [J2,J3]=J1, [J3,J1]=J2 -> {}", py_bools(&rot_rot));

fn echelle(M: &Vec<Vec<Fract>>, c: Fract) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); M[0].len()]; M.len()];
    for i in 0..M.len() { for j in 0..M[0].len() { R[i][j] = M[i][j] * c; } }
    R
}
let mut melange_ok = true;
for i in 1..=3 {
    for j in 1..=3 {
        let mut attendu = zeros(4);
        for k in 1..=3 {
            let e = eps(i, j, k);
            if e != 0 {
                attendu = add(&attendu, &echelle(&base[3 + k - 1], Fract::int(e as i128)));
            }
        }
        if !eq(&br(&base[i - 1], &base[3 + j - 1]), &attendu) {
            melange_ok = false;
        }
    }
}
println!("[Ji,Kj] = eps_ijk Kk (les 9 crochets) -> {}", py_bool(melange_ok));
let boost_boost = vec![
    eq(&br(&K1, &K2), &neg(&J3)),
    eq(&br(&K2, &K3), &neg(&J1)),
    eq(&br(&K3, &K1), &neg(&J2)),
];
println!("[K1,K2]=-J3, [K2,K3]=-J1, [K3,K1]=-J2 -> {}", py_bools(&boost_boost));

[J1,J2]=J3, [J2,J3]=J1, [J3,J1]=J2 -> [True, True, True]


[Ji,Kj] = eps_ijk Kk (les 9 crochets) -> True


[K1,K2]=-J3, [K2,K3]=-J1, [K3,K1]=-J2 -> [True, True, True]


### Lecture du résultat (constantes de structure)

**Mesuré** : l'éq. 80 exacte sur notre modèle — les rotations se ferment sur elles-mêmes, les crochets rotation-boost **font tourner le boost** (les neuf $[J_i, K_j] = \epsilon_{ijk} K_k$), et deux boosts donnent **moins une rotation**. « at every structure constant it equals the Lorentz algebra of the literature by name » : ici ce sont nos propres fractions qui font le compte.

In [4]:
// L'identite de Jacobi sur les 216 triples de la base.
use std::cmp::Ordering;
use std::fmt;
use std::ops::{Add, Sub, Mul, Div, Neg};

#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: u128, mut b: u128) -> u128 {
    while b != 0 { let t = a % b; a = b; b = t; } a
}

impl Fract {
    fn new(n: i128, d: i128) -> Self {
        assert!(d != 0);
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = gcd(n.unsigned_abs(), d as u128) as i128;
        Self { num: n / g, den: d / g }
    }
    fn int(n: i128) -> Self { Self::new(n, 1) }
}

impl Add for Fract { type Output=Self; fn add(self,r:Self)->Self { Self::new(self.num*r.den+r.num*self.den,self.den*r.den) } }
impl Sub for Fract { type Output=Self; fn sub(self,r:Self)->Self { Self::new(self.num*r.den-r.num*self.den,self.den*r.den) } }
impl Mul for Fract { type Output=Self; fn mul(self,r:Self)->Self { Self::new(self.num*r.num,self.den*r.den) } }
impl Div for Fract { type Output=Self; fn div(self,r:Self)->Self { Self::new(self.num*r.den,self.den*r.num) } }
impl Neg for Fract { type Output=Self; fn neg(self)->Self { Self::new(-self.num,self.den) } }
impl PartialOrd for Fract { fn partial_cmp(&self,r:&Self)->Option<Ordering> { (self.num*r.den).partial_cmp(&(r.num*self.den)) } }
impl fmt::Display for Fract {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

fn mul(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let n = A.len(); let m = B[0].len(); let k = B.len();
    let mut R = vec![vec![Fract::int(0); m]; n];
    for i in 0..n { for j in 0..m {
        let mut s = Fract::int(0);
        for t in 0..k { s = s + A[i][t] * B[t][j]; }
        R[i][j] = s;
    }}
    R
}
fn add(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = A[i][j] + B[i][j]; } }
    R
}
fn sub(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = A[i][j] - B[i][j]; } }
    R
}
fn br(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    sub(&mul(A, B), &mul(B, A))
}
fn eq(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> bool {
    A.iter().zip(B.iter()).all(|(ra, rb)| ra.iter().zip(rb.iter()).all(|(a, b)| a == b))
}
fn transp(A: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let n = A.len();
    let mut R = vec![vec![Fract::int(0); n]; n];
    for i in 0..n { for j in 0..n { R[i][j] = A[j][i]; } }
    R
}
fn neg(A: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = -A[i][j]; } }
    R
}
fn zeros(n: usize) -> Vec<Vec<Fract>> {
    vec![vec![Fract::int(0); n]; n]
}
fn E(i: usize, j: usize) -> Vec<Vec<Fract>> {
    let mut M = zeros(4); M[i][j] = Fract::int(1); M
}
fn nulle(A: &Vec<Vec<Fract>>) -> bool {
    A.iter().all(|row| row.iter().all(|&v| v == Fract::int(0)))
}

let J1 = sub(&E(3, 2), &E(2, 3));
let J2 = sub(&E(1, 3), &E(3, 1));
let J3 = sub(&E(2, 1), &E(1, 2));
fn boost(i: usize) -> Vec<Vec<Fract>> {
    let mut K = zeros(4);
    K[0][i] = Fract::new(1, 2);
    K[i][0] = Fract::int(2);
    K
}
let K1 = boost(1);
let K2 = boost(2);
let K3 = boost(3);
let base = vec![J1, J2, J3, K1, K2, K3];

let total = base.len().pow(3);
let mut jacobi_reussis = 0;
for X in &base {
    for Y in &base {
        for Z in &base {
            let j = add(&add(&br(&br(X, Y), Z), &br(&br(Y, Z), X)), &br(&br(Z, X), Y));
            if nulle(&j) {
                jacobi_reussis += 1;
            }
        }
    }
}
fn py_bool(b: bool) -> String {
    if b { "True".to_string() } else { "False".to_string() }
}
println!("Jacobi {}/{} triples -> {}", jacobi_reussis, total, py_bool(jacobi_reussis == total));

Jacobi 216/216 triples -> True


### Lecture du résultat

**Mesuré** : l'identité de Jacobi tient sur les 216 triples de la base — le crochet est une structure d'algèbre de Lie, pas seulement quinze égalités isolées.

In [5]:
// Forme de Killing k(X,Y) = Tr(ad_X ad_Y) dans la base (J1..J3, K1..K3).
use std::cmp::Ordering;
use std::fmt;
use std::ops::{Add, Sub, Mul, Div, Neg};

#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: u128, mut b: u128) -> u128 {
    while b != 0 { let t = a % b; a = b; b = t; } a
}

impl Fract {
    fn new(n: i128, d: i128) -> Self {
        assert!(d != 0);
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = gcd(n.unsigned_abs(), d as u128) as i128;
        Self { num: n / g, den: d / g }
    }
    fn int(n: i128) -> Self { Self::new(n, 1) }
}

impl Add for Fract { type Output=Self; fn add(self,r:Self)->Self { Self::new(self.num*r.den+r.num*self.den,self.den*r.den) } }
impl Sub for Fract { type Output=Self; fn sub(self,r:Self)->Self { Self::new(self.num*r.den-r.num*self.den,self.den*r.den) } }
impl Mul for Fract { type Output=Self; fn mul(self,r:Self)->Self { Self::new(self.num*r.num,self.den*r.den) } }
impl Div for Fract { type Output=Self; fn div(self,r:Self)->Self { Self::new(self.num*r.den,self.den*r.num) } }
impl Neg for Fract { type Output=Self; fn neg(self)->Self { Self::new(-self.num,self.den) } }
impl PartialOrd for Fract { fn partial_cmp(&self,r:&Self)->Option<Ordering> { (self.num*r.den).partial_cmp(&(r.num*self.den)) } }
impl fmt::Display for Fract {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

fn mul(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let n = A.len(); let m = B[0].len(); let k = B.len();
    let mut R = vec![vec![Fract::int(0); m]; n];
    for i in 0..n { for j in 0..m {
        let mut s = Fract::int(0);
        for t in 0..k { s = s + A[i][t] * B[t][j]; }
        R[i][j] = s;
    }}
    R
}
fn add(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = A[i][j] + B[i][j]; } }
    R
}
fn sub(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = A[i][j] - B[i][j]; } }
    R
}
fn br(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    sub(&mul(A, B), &mul(B, A))
}
fn eq(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> bool {
    A.iter().zip(B.iter()).all(|(ra, rb)| ra.iter().zip(rb.iter()).all(|(a, b)| a == b))
}
fn transp(A: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let n = A.len();
    let mut R = vec![vec![Fract::int(0); n]; n];
    for i in 0..n { for j in 0..n { R[i][j] = A[j][i]; } }
    R
}
fn neg(A: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = -A[i][j]; } }
    R
}
fn zeros(n: usize) -> Vec<Vec<Fract>> {
    vec![vec![Fract::int(0); n]; n]
}
fn E(i: usize, j: usize) -> Vec<Vec<Fract>> {
    let mut M = zeros(4); M[i][j] = Fract::int(1); M
}
fn nulle(A: &Vec<Vec<Fract>>) -> bool {
    A.iter().all(|row| row.iter().all(|&v| v == Fract::int(0)))
}

let J1 = sub(&E(3, 2), &E(2, 3));
let J2 = sub(&E(1, 3), &E(3, 1));
let J3 = sub(&E(2, 1), &E(1, 2));
fn boost(i: usize) -> Vec<Vec<Fract>> {
    let mut K = zeros(4);
    K[0][i] = Fract::new(1, 2);
    K[i][0] = Fract::int(2);
    K
}
let K1 = boost(1);
let K2 = boost(2);
let K3 = boost(3);
let base = vec![J1, J2, J3, K1, K2, K3];

fn coeffs(C: &Vec<Vec<Fract>>, base: &Vec<Vec<Vec<Fract>>>) -> Option<Vec<i64>> {
    if nulle(C) {
        return Some(vec![0; 6]);
    }
    for k in 0..6 {
        if eq(C, &base[k]) {
            let mut v = vec![0i64; 6]; v[k] = 1; return Some(v);
        }
        if eq(C, &neg(&base[k])) {
            let mut v = vec![0i64; 6]; v[k] = -1; return Some(v);
        }
    }
    None
}

let mut ads: Vec<Vec<Vec<i64>>> = Vec::new();
let mut fermeture = true;
for X in &base {
    let mut cols = Vec::new();
    for e in &base {
        match coeffs(&br(X, e), &base) {
            Some(v) => cols.push(v),
            None => { fermeture = false; cols.push(vec![0; 6]); }
        }
    }
    ads.push(cols);
}

let mut kappa = vec![vec![Fract::int(0); 6]; 6];
for a in 0..6 {
    for b in 0..6 {
        let mut s = Fract::int(0);
        for j in 0..6 {
            for k in 0..6 {
                s = s + Fract::int(ads[a][j][k] as i128) * Fract::int(ads[b][k][j] as i128);
            }
        }
        kappa[a][b] = s;
    }
}

fn signature_sym(S: &Vec<Vec<Fract>>) -> (usize, usize) {
    let mut M: Vec<Vec<Fract>> = S.iter().map(|row| row.to_vec()).collect();
    let mut signes: Vec<i64> = Vec::new();
    for _ in 0..S.len() {
        let mut pr: Option<usize> = None;
        let mut best_q = Fract::int(0);
        for r in 0..M.len() {
            let a = if M[r][r] < Fract::int(0) { -M[r][r] } else { M[r][r] };
            if a > best_q {
                best_q = a;
                pr = Some(r);
            }
        }
        match pr {
            None => break,
            Some(p) => {
                if M[p][p] == Fract::int(0) { break; }
                signes.push(if M[p][p] > Fract::int(0) { 1 } else { -1 });
                let pivot = M[p][p];
                let garde: Vec<usize> = (0..M.len()).filter(|&i| i != p).collect();
                let mut newM = Vec::new();
                for i in &garde {
                    let mut row = Vec::new();
                    for j in &garde {
                        row.push(M[*i][*j] - M[*i][p] * M[p][*j] / pivot);
                    }
                    newM.push(row);
                }
                M = newM;
            }
        }
    }
    (signes.iter().filter(|&&s| s == 1).count(), signes.iter().filter(|&&s| s == -1).count())
}

let sig_kappa = signature_sym(&kappa);
let kappa_diag_ok = (0..6).all(|i| kappa[i][i] == (if i < 3 { Fract::int(-4) } else { Fract::int(4) }));

fn py_bool(b: bool) -> String {
    if b { "True".to_string() } else { "False".to_string() }
}

println!("fermeture de la base -> {}", py_bool(fermeture));
let diag: Vec<String> = (0..6).map(|i| format!("{}", kappa[i][i])).collect();
println!("kappa diagonale (J1..J3, K1..K3) : [{}]", diag.join(", "));
let hors_diag_ok = (0..6).all(|i| (0..6).all(|j| if i != j { kappa[i][j] == Fract::int(0) } else { true }));
println!("kappa hors-diagonale nulle -> {}", py_bool(hors_diag_ok));
println!("signature de kappa (pivots symetriques) : ({}, {}) (attendu (3, 3)) ; diagonale (-4,-4,-4,4,4,4) : {}", sig_kappa.0, sig_kappa.1, py_bool(kappa_diag_ok));

fermeture de la base -> True


kappa diagonale (J1..J3, K1..K3) : [-4, -4, -4, 4, 4, 4]


kappa hors-diagonale nulle -> True


signature de kappa (pivots symetriques) : (3, 3) (attendu (3, 3)) ; diagonale (-4,-4,-4,4,4,4) : True


### Lecture du résultat (Killing)

**Mesuré** : $\kappa(J,J) = -4$ pour chaque rotation, $\kappa(K,K) = +4$ pour chaque boost, tous les termes croisés nuls — la forme de Killing est $\mathrm{diag}(-4,-4,-4,\,4,4,4)$, signature **(3, 3)** par élimination symétrique exacte. Les sous-espaces se séparent d'eux-mêmes : les rotations portent (0, 3), les boosts (3, 0) — et la moitié (0, 3) est précisément la lecture du contre-test du source (« with the unit struck the six-dimensional algebra moves to the three-dimensional isometry algebra of the directions alone at the signature zero and three », p. 145, cité).

## Les Grams des deux liens : l'éq. 79 et l'éq. 81

À l'unité hermitienne du lien qutrit-qubit (dimension six) : « the unit squaring to the link dimension six, every direction step to minus three halves and every interference term zero » (p. 145) — l'éq. 79, $G = \tfrac32\,\mathrm{diag}(4,-1,-1,-1)$. À la chaîne série (2,2), la Gram de propagation (éq. 81) est $\mathrm{diag}(4,-1,-1,-1)$, signature $(1,3)$, le rapport $-4$.

**Le rapport est le même aux deux types de lien** : $6 \div \tfrac32 = 4 \div 1 = 4$ — l'unité pèse quatre directions, quel que soit le lien. Cité : la Gram calculée cellule par cellule aux **six types de paire**, la même scission partout.

In [6]:
// Eq. 79 : G = diag(6, -3/2, -3/2, -3/2) = (3/2)*diag(4,-1,-1,-1) ; eq. 81 : diag(4,-1,-1,-1).
use std::cmp::Ordering;
use std::fmt;
use std::ops::{Add, Sub, Mul, Div, Neg};

#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: u128, mut b: u128) -> u128 {
    while b != 0 { let t = a % b; a = b; b = t; } a
}

impl Fract {
    fn new(n: i128, d: i128) -> Self {
        assert!(d != 0);
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = gcd(n.unsigned_abs(), d as u128) as i128;
        Self { num: n / g, den: d / g }
    }
    fn int(n: i128) -> Self { Self::new(n, 1) }
}

impl Add for Fract { type Output=Self; fn add(self,r:Self)->Self { Self::new(self.num*r.den+r.num*self.den,self.den*r.den) } }
impl Sub for Fract { type Output=Self; fn sub(self,r:Self)->Self { Self::new(self.num*r.den-r.num*self.den,self.den*r.den) } }
impl Mul for Fract { type Output=Self; fn mul(self,r:Self)->Self { Self::new(self.num*r.num,self.den*r.den) } }
impl Div for Fract { type Output=Self; fn div(self,r:Self)->Self { Self::new(self.num*r.den,self.den*r.num) } }
impl Neg for Fract { type Output=Self; fn neg(self)->Self { Self::new(-self.num,self.den) } }
impl PartialOrd for Fract { fn partial_cmp(&self,r:&Self)->Option<Ordering> { (self.num*r.den).partial_cmp(&(r.num*self.den)) } }
impl fmt::Display for Fract {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

fn py_bool(b: bool) -> String {
    if b { "True".to_string() } else { "False".to_string() }
}

let G79 = vec![Fract::int(6), Fract::new(-3, 2), Fract::new(-3, 2), Fract::new(-3, 2)];
let G81 = vec![Fract::int(4), Fract::int(-1), Fract::int(-1), Fract::int(-1)];

let eq79_ok = (0..4).all(|i| Fract::new(3, 2) * G81[i] == G79[i]);
println!("(3/2)*diag(4,-1,-1,-1) = eq.79 -> {}", py_bool(eq79_ok));

let ratio79 = G79[0] / G79[1];
let ratio81 = G81[0] / G81[1];
println!("ratio eq.79 : 6 / (-3/2) = {}  ; ratio eq.81 : 4 / (-1) = {}", ratio79, ratio81);

fn sig_pivots(diag: &Vec<Fract>) -> (usize, usize) {
    let s: Vec<i64> = diag.iter().map(|v| if *v > Fract::int(0) { 1 } else { -1 }).collect();
    (s.iter().filter(|&&x| x == 1).count(), s.iter().filter(|&&x| x == -1).count())
}

fn sig_sylvester(diag: &Vec<Fract>) -> (usize, usize) {
    let mut prods: Vec<Fract> = Vec::new();
    let mut p = Fract::int(1);
    for v in diag {
        p = p * *v;
        prods.push(p);
    }
    let mut suite: Vec<i64> = vec![1];
    for x in &prods {
        suite.push(if *x > Fract::int(0) { 1 } else { -1 });
    }
    let negatifs = (0..suite.len() - 1).filter(|&i| suite[i] != suite[i + 1]).count();
    (diag.len() - negatifs, negatifs)
}

for (nom, G) in [("eq.79", &G79), ("eq.81", &G81)] {
    let sp = sig_pivots(G);
    let ss = sig_sylvester(G);
    println!("{} : signes pivots ({}, {}) ; Sylvester ({}, {})", nom, sp.0, sp.1, ss.0, ss.1);
};

(3/2)*diag(4,-1,-1,-1) = eq.79 -> True


ratio eq.79 : 6 / (-3/2) = -4  ; ratio eq.81 : 4 / (-1) = -4


eq.79 : signes pivots (1, 3) ; Sylvester (1, 3)


eq.81 : signes pivots (1, 3) ; Sylvester (1, 3)


### Lecture du résultat (Grams)

**Mesuré** : les deux Grams donnent la signature (1, 3) par les **deux méthodes des deux moteurs du source** — les signes des pivots (« the pivot signs at the first engine ») et les mineurs principaux (« the leading principal minors at the second », critère de Sylvester) — et le rapport $-4$ est commun aux deux liens. **Cité** : le calcul cellule par cellule, l'égalité à la rangée (2,2) aux deux bouts, 116/116 pas (§55).

## §55 — Théorème 55.1 : un contre trois à chaque pas de la chaîne

« At every chain and step carrying a (2,2) link, the Gram of the propagation reads one against three, the boost of the chain is its isometry with the mixing block nonzero, and the one-unit rotation [...] is the isometry of a definite plane and of no restriction of that Gram » (p. 146). **Standing: theorem.**

Le census (cité) : 34/36 états de chaîne parcourus à 116 pas sur 82 liens (2,2) série des neuf toiles enregistrées — l'électron, le muon, le tau, le proton, le neutron jusqu'au deutéron, le mot photon, la toile univers, le puits neutron et le puits tau. Nous, nous mesurons les **matrices imprimées** du plan : les deux générateurs, leurs deux plans, leurs carrés et leurs résidus exacts.

In [7]:
// Le plan (eq. 82 et figure 28) : l'unite hermitienne et une direction portee.
use std::cmp::Ordering;
use std::fmt;
use std::ops::{Add, Sub, Mul, Div, Neg};

#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: u128, mut b: u128) -> u128 {
    while b != 0 { let t = a % b; a = b; b = t; } a
}

impl Fract {
    fn new(n: i128, d: i128) -> Self {
        assert!(d != 0);
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = gcd(n.unsigned_abs(), d as u128) as i128;
        Self { num: n / g, den: d / g }
    }
    fn int(n: i128) -> Self { Self::new(n, 1) }
}

impl Add for Fract { type Output=Self; fn add(self,r:Self)->Self { Self::new(self.num*r.den+r.num*self.den,self.den*r.den) } }
impl Sub for Fract { type Output=Self; fn sub(self,r:Self)->Self { Self::new(self.num*r.den-r.num*self.den,self.den*r.den) } }
impl Mul for Fract { type Output=Self; fn mul(self,r:Self)->Self { Self::new(self.num*r.num,self.den*r.den) } }
impl Div for Fract { type Output=Self; fn div(self,r:Self)->Self { Self::new(self.num*r.den,self.den*r.num) } }
impl Neg for Fract { type Output=Self; fn neg(self)->Self { Self::new(-self.num,self.den) } }
impl PartialOrd for Fract { fn partial_cmp(&self,r:&Self)->Option<Ordering> { (self.num*r.den).partial_cmp(&(r.num*self.den)) } }
impl fmt::Display for Fract {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

fn mul(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let n = A.len(); let m = B[0].len(); let k = B.len();
    let mut R = vec![vec![Fract::int(0); m]; n];
    for i in 0..n { for j in 0..m {
        let mut s = Fract::int(0);
        for t in 0..k { s = s + A[i][t] * B[t][j]; }
        R[i][j] = s;
    }}
    R
}
fn add(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = A[i][j] + B[i][j]; } }
    R
}
fn sub(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = A[i][j] - B[i][j]; } }
    R
}
fn eq(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> bool {
    A.iter().zip(B.iter()).all(|(ra, rb)| ra.iter().zip(rb.iter()).all(|(a, b)| a == b))
}
fn transp(A: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let n = A.len();
    let mut R = vec![vec![Fract::int(0); n]; n];
    for i in 0..n { for j in 0..n { R[i][j] = A[j][i]; } }
    R
}
fn neg(A: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = -A[i][j]; } }
    R
}

fn py_bool(b: bool) -> String {
    if b { "True".to_string() } else { "False".to_string() }
}

fn m2(M: &Vec<Vec<Fract>>) -> String {
    let rows: Vec<String> = M.iter().map(|row| {
        row.iter().map(|v| format!("{}", v)).collect::<Vec<_>>().join(", ")
    }).collect();
    format!("[{}]", rows.join("; "))
}

// le plan 4, 0 ; 0, -1 (indefini)
let PL = vec![vec![Fract::int(4), Fract::int(0)], vec![Fract::int(0), Fract::int(-1)]];
// le plan des ordres 1, 0 ; 0, 1/4 (defini)
let Dp = vec![vec![Fract::int(1), Fract::int(0)], vec![Fract::int(0), Fract::new(1, 4)]];
// le boost 0, 1/2 ; 2, 0
let Kp = vec![vec![Fract::int(0), Fract::new(1, 2)], vec![Fract::int(2), Fract::int(0)]];
// la rotation d'unite 0, -1/2 ; 2, 0
let Rp = vec![vec![Fract::int(0), Fract::new(-1, 2)], vec![Fract::int(2), Fract::int(0)]];
let I2 = vec![vec![Fract::int(1), Fract::int(0)], vec![Fract::int(0), Fract::int(1)]];
let Z2 = vec![vec![Fract::int(0), Fract::int(0)], vec![Fract::int(0), Fract::int(0)]];

fn resid2(A: &Vec<Vec<Fract>>, P: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    add(&mul(&transp(A), P), &mul(P, A))
}

let r_boost = resid2(&Kp, &PL);
let boost_ok = eq(&r_boost, &Z2);
println!("K^T PL + PL K = {} -> nul : {}", m2(&r_boost), py_bool(boost_ok));

let k_carre = eq(&mul(&Kp, &Kp), &I2);
println!("K^2 = I -> {}", py_bool(k_carre));

let r_carre = eq(&mul(&Rp, &Rp), &neg(&I2));
println!("R^2 = -I -> {}", py_bool(r_carre));

let r_eucl = resid2(&Rp, &Dp);
let r_eucl_ok = eq(&r_eucl, &Z2);
println!("R^T D + D R = {} -> nul (euclidien) : {}", m2(&r_eucl), py_bool(r_eucl_ok));

let r_gram = resid2(&Rp, &PL);
let r_gram_ok = eq(&r_gram, &vec![vec![Fract::int(0), Fract::int(-4)], vec![Fract::int(-4), Fract::int(0)]]);
println!("R^T PL + PL R = {} -> attendu [[0, -4], [-4, 0]] : {} ; non nul : pas une isometrie de la Gram", m2(&r_gram), py_bool(r_gram_ok));

K^T PL + PL K = [0, 0; 0, 0] -> nul : True


K^2 = I -> True


R^2 = -I -> True


R^T D + D R = [0, 0; 0, 0] -> nul (euclidien) : True


R^T PL + PL R = [0, -4; -4, 0] -> attendu [[0, -4], [-4, 0]] : True ; non nul : pas une isometrie de la Gram


### Lecture du résultat (le plan, ligne à ligne de la figure 28)

**Mesuré** : le boost $\begin{psmallmatrix} 0 & 1/2 \\ 2 & 0 \end{psmallmatrix}$ a un résidu nul sur $\mathrm{diag}(4,-1)$ et **carré l'identité** ; la rotation d'unité $\begin{psmallmatrix} 0 & -1/2 \\ 2 & 0 \end{psmallmatrix}$ a **carré moins l'identité**, résidu nul sur le plan défini $\mathrm{diag}(1, \tfrac14)$ (l'appariement euclidien), et sur la Gram restreinte son résidu est exactement $\begin{psmallmatrix} 0 & -4 \\ -4 & 0 \end{psmallmatrix}$ — non nul : **pas une isométrie**. La table de la figure 28 (p. 150), reproduite colonne par colonne : générateur, plan, carré, appariement préservé, résidu, lettre — Euclidien pour la rotation, Lorentzien pour le boost, aucune isométrie pour la restriction.

In [8]:
// Le releve au bout qubit : le triple litteral du ch. 55 (deja croise au ch. 49).
use std::cmp::Ordering;
use std::fmt;
use std::ops::{Add, Sub, Mul, Div, Neg};

#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: u128, mut b: u128) -> u128 {
    while b != 0 { let t = a % b; a = b; b = t; } a
}

impl Fract {
    fn new(n: i128, d: i128) -> Self {
        assert!(d != 0);
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = gcd(n.unsigned_abs(), d as u128) as i128;
        Self { num: n / g, den: d / g }
    }
    fn int(n: i128) -> Self { Self::new(n, 1) }
}

impl Add for Fract { type Output=Self; fn add(self,r:Self)->Self { Self::new(self.num*r.den+r.num*self.den,self.den*r.den) } }
impl Sub for Fract { type Output=Self; fn sub(self,r:Self)->Self { Self::new(self.num*r.den-r.num*self.den,self.den*r.den) } }
impl Mul for Fract { type Output=Self; fn mul(self,r:Self)->Self { Self::new(self.num*r.num,self.den*r.den) } }
impl Div for Fract { type Output=Self; fn div(self,r:Self)->Self { Self::new(self.num*r.den,self.den*r.num) } }
impl Neg for Fract { type Output=Self; fn neg(self)->Self { Self::new(-self.num,self.den) } }
impl PartialOrd for Fract { fn partial_cmp(&self,r:&Self)->Option<Ordering> { (self.num*r.den).partial_cmp(&(r.num*self.den)) } }
impl fmt::Display for Fract {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct C { re: Fract, im: Fract }

fn cscale(z: C, n: Fract) -> C {
    C { re: z.re * n, im: z.im * n }
}

let C0 = C { re: Fract::int(0), im: Fract::int(0) };
let CI = C { re: Fract::int(0), im: Fract::int(1) };

let G1 = vec![
    vec![C0, cscale(CI, Fract::new(-1, 2))],
    vec![cscale(CI, Fract::new(-1, 2)), C0],
];
let G2 = vec![
    vec![C0, C { re: Fract::new(1, 2), im: Fract::int(0) }],
    vec![C { re: Fract::new(-1, 2), im: Fract::int(0) }, C0],
];
let G3 = vec![
    vec![cscale(CI, Fract::new(-1, 2)), C0],
    vec![C0, cscale(CI, Fract::new(1, 2))],
];

// re-orientation (flip G2)
let G2_flip: Vec<Vec<C>> = G2.iter().map(|row| row.iter().map(|z| C { re: -z.re, im: -z.im }).collect()).collect();
let T = vec![G1, G2_flip, G3];

fn dagger(M: &Vec<Vec<C>>) -> Vec<Vec<C>> {
    let mut R = vec![vec![C { re: Fract::int(0), im: Fract::int(0) }; 2]; 2];
    for i in 0..2 {
        for j in 0..2 {
            R[i][j] = C { re: M[j][i].re, im: -M[j][i].im };
        }
    }
    R
}

fn meq2(A: &Vec<Vec<C>>, B: &Vec<Vec<C>>) -> bool {
    for i in 0..2 { for j in 0..2 { if A[i][j] != B[i][j] { return false; } } }
    true
}

fn negc(M: &Vec<Vec<C>>) -> Vec<Vec<C>> {
    M.iter().map(|row| row.iter().map(|z| C { re: -z.re, im: -z.im }).collect()).collect()
}

let anti: Vec<bool> = T.iter().map(|t| meq2(&dagger(t), &negc(t))).collect();

fn py_bools(v: &Vec<bool>) -> String {
    let parts: Vec<String> = v.iter().map(|&b| if b { "True".to_string() } else { "False".to_string() }).collect();
    format!("[{}]", parts.join(", "))
}
fn py_bool(b: bool) -> String {
    if b { "True".to_string() } else { "False".to_string() }
}

println!("les trois rotations anti-hermitiennes (dagger = -M) : {} -> {}", py_bools(&anti), py_bool(anti.iter().all(|&b| b)));
println!("les trois boosts hermitiens au bout qubit : CITE -- le releve est celui de la");
println!("construction ; notre generateur de plan porte l'isometrie eta, mesurée plus haut");

les trois rotations anti-hermitiennes (dagger = -M) : [True, True, True] -> True


les trois boosts hermitiens au bout qubit : CITE -- le releve est celui de la


construction ; notre generateur de plan porte l'isometrie eta, mesurée plus haut


### Lecture du résultat (le relevé)

**Mesuré** : les trois générateurs de rotation du triple littéral — celui-là même que la chaîne transporte (« the transposed triple », p. 147) — sont anti-hermitiens au dagger ordinaire : les « three anti-Hermitian » du théorème. **Cité** : les « three Hermitian generators » du côté boost — le relevé du bout qubit appartient à la construction, notre modèle porte l'isométrie $\eta$.

In [9]:
// Le cone nul du plan 4, 0 ; 0, -1, et le contraste Pell (rapport non carre).
use std::cmp::Ordering;
use std::fmt;
use std::ops::{Add, Sub, Mul, Div, Neg};

#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: u128, mut b: u128) -> u128 {
    while b != 0 { let t = a % b; a = b; b = t; } a
}

impl Fract {
    fn new(n: i128, d: i128) -> Self {
        assert!(d != 0);
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = gcd(n.unsigned_abs(), d as u128) as i128;
        Self { num: n / g, den: d / g }
    }
    fn int(n: i128) -> Self { Self::new(n, 1) }
}

impl Add for Fract { type Output=Self; fn add(self,r:Self)->Self { Self::new(self.num*r.den+r.num*self.den,self.den*r.den) } }
impl Sub for Fract { type Output=Self; fn sub(self,r:Self)->Self { Self::new(self.num*r.den-r.num*self.den,self.den*r.den) } }
impl Mul for Fract { type Output=Self; fn mul(self,r:Self)->Self { Self::new(self.num*r.num,self.den*r.den) } }
impl Div for Fract { type Output=Self; fn div(self,r:Self)->Self { Self::new(self.num*r.den,self.den*r.num) } }
impl Neg for Fract { type Output=Self; fn neg(self)->Self { Self::new(-self.num,self.den) } }
impl PartialOrd for Fract { fn partial_cmp(&self,r:&Self)->Option<Ordering> { (self.num*r.den).partial_cmp(&(r.num*self.den)) } }
impl fmt::Display for Fract {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

let PL = vec![vec![Fract::int(4), Fract::int(0)], vec![Fract::int(0), Fract::int(-1)]];

let mut cone_ok = true;
for v in [(1i64, 2i64), (1i64, -2i64)] {
    let q = PL[0][0] * Fract::int(v.0 as i128) * Fract::int(v.0 as i128) + PL[1][1] * Fract::int(v.1 as i128) * Fract::int(v.1 as i128);
    if q != Fract::int(0) {
        cone_ok = false;
    }
    println!("direction ({}, {}) : 4t^2 - x^2 = {}", v.0, v.1, q);
}
let pell: i64 = 3*3 - 2*2*2;
println!("Pell : 3^2 - 2*2^2 = {} -- la solution fondamentale de t^2 - 2x^2 = 1 (Chaines-09)", pell);

direction (1, 2) : 4t^2 - x^2 = 0


direction (1, -2) : 4t^2 - x^2 = 0


Pell : 3^2 - 2*2^2 = 1 -- la solution fondamentale de t^2 - 2x^2 = 1 (Chaines-09)


### Lecture du résultat (le cône et Pell)

**Mesuré** : le plan de l'unité porte exactement deux directions nulles, $(1, 2)$ et $(1, -2)$ — le cône dessiné au-dessus de la chaîne (figure 27) ; et $3^2 - 2\cdot2^2 = 1$, la solution fondamentale de Pell. **Carrefour** : la famille de Pell donne une **infinité** de cartes au rapport non carré (Chaînes-09), quand le rapport carré donne un groupe **fini** — les 24 rotations du réseau de Chaînes-10. « while the Euclidean pairing admits a finite group at every ratio » (p. 145, cité) : nos deux carnets précédents en sont les deux moitiés.

## Ce que le census porte (§54, cité)

6 objets énumérés ; la Gram calculée cellule par cellule aux **six types de paire**, la scission (1, 3) partout ; le contre-test de l'unité frappée — l'algèbre de dimension six passe à l'algèbre de directions de dimension trois à la signature (0, 3), « with no plane of the unit » ; cinq contre-tests aux lettres épinglées ; treize contrôles lus pareil aux deux moteurs ; la reproduction par une seconde construction, égale à chaque cellule ; la plus longue chaîne de prémisses : douze chapitres.

In [10]:
// Les comptes du census (§54), cites tels qu'imprimes (p. 145-146).
let census_54: Vec<(&str, i64)> = vec![
    ("objets enumeres", 6),
    ("types de paire (Gram cellule par cellule)", 6),
    ("contre-tests aux lettres epinglees", 5),
    ("controles lus pareil aux deux moteurs", 13),
    ("longueur de la plus longue chaine de premices (chapitres)", 12),
];
for (k, v) in &census_54 {
    println!("{} : {} (CITE)", k, v);
}
println!("contre-test nomme : unite frappee -> algebre de directions seule, dimension 3,");
println!("signature (0, 3), sans plan de l'unite (cite) ; notre moitie negative de kappa la lit");

objets enumeres : 6 (CITE)


types de paire (Gram cellule par cellule) : 6 (CITE)


contre-tests aux lettres epinglees : 5 (CITE)


controles lus pareil aux deux moteurs : 13 (CITE)


longueur de la plus longue chaine de premices (chapitres) : 12 (CITE)


contre-test nomme : unite frappee -> algebre de directions seule, dimension 3,


signature (0, 3), sans plan de l'unite (cite) ; notre moitie negative de kappa la lit


## Ce que le census porte (§55, cité)

Un contre trois à **116/116** pas (défini à aucun, autre à aucun) ; le boost isométrie à 116/116 ; la rotation d'unité isométrie de son plan défini à 116/116 et **d'aucune restriction de la Gram** ; l'orientation de la triple portée **préservée aux pas pairs, renversée aux impairs** (50/116) — figure 26 : a1 p, x1 r, x2 p, x3 r, a2 p sur la chaîne de l'électron ; les constantes de structure **niées aux pas impairs** à la base littérale portée (78/90 au premier site inséré, les zéros tenant) et égales à la table des crochets à **90/90** à la base ré-orientée à chaque pas — c'est le flip de Chaînes-10 qui alterne avec la parité ; le pôle à chaque lien, fonction à deux points $(30+30\sqrt2)/2304$, 204/204 ; le résultat de boost ouvert transcrit intégralement et re-dérivé au cadre du lien (2,3), les deux comptes imprimés par la cellule de census qui suit.

Quatre contre-tests : l'unité frappée → le plan devient défini et le type rotation à chaque pas ; l'orientation renversée à chaque pas → les comptes bougent, la signature pas ; le lien (2,3) terminal du proton et du photon lu comme lien de chaîne → la Gram passe à $\mathrm{diag}(6,-1,-1,-1)$ à ces deux pas, signature inchangée ; la porte frappée → toutes les cellules absentes.

In [11]:
// Les comptes du census (§55), cites tels qu'imprimes (p. 146-149).
let census_55: Vec<(&str, &str)> = vec![
    ("etats de chaine parcourus", "34 / 36"),
    ("pas marches", "116"),
    ("liens (2,2) serie", "82"),
    ("un contre trois", "116 / 116 (defini a aucun, autre a aucun)"),
    ("boost isometrie", "116 / 116"),
    ("rotation d'unite sur son plan defini", "116 / 116"),
    ("orientation preservee (pairs) / renversee (impairs)", "50 / 116"),
    ("constantes egales a la table (base re-orientee)", "90 / 90 a 116/116"),
    ("pole a deux points", "204 / 204"),
    ("resultat de boost ouvert transcrit / re-derive", "15 / 15 et 10 / 10"),
];
for (k, v) in &census_55 {
    println!("{} : {} (CITE)", k, v);
}

fn py_str_list(v: &Vec<&str>) -> String {
    let parts: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", parts.join(", "))
}

let lettres: Vec<&str> = vec!["a1 p", "x1 r", "x2 p", "x3 r", "a2 p"];
println!("orientation de la chaine de l'electron (figure 26, citee) : {}", py_str_list(&lettres));

etats de chaine parcourus : 34 / 36 (CITE)


pas marches : 116 (CITE)


liens (2,2) serie : 82 (CITE)


un contre trois : 116 / 116 (defini a aucun, autre a aucun) (CITE)


boost isometrie : 116 / 116 (CITE)


rotation d'unite sur son plan defini : 116 / 116 (CITE)


orientation preservee (pairs) / renversee (impairs) : 50 / 116 (CITE)


constantes egales a la table (base re-orientee) : 90 / 90 a 116/116 (CITE)


pole a deux points : 204 / 204 (CITE)


resultat de boost ouvert transcrit / re-derive : 15 / 15 et 10 / 10 (CITE)


orientation de la chaine de l'electron (figure 26, citee) : ['a1 p', 'x1 r', 'x2 p', 'x3 r', 'a2 p']


## Ouvert — ligne à ligne

1. **L'unité de boost ouverte au ch. 56** : « the one unit open at chapter 56 » — le prochain carnet (Théorème 56.2, open result, et la famille de Pell à l'unité debout).
2. **Le résultat de boost ouvert** dont le 55.1 transcrit les cellules : ses prémisses (l'ouvert 92.1, les théorèmes 92.2 et 115.1) ne sont pas prouvées ici.
3. **Le pôle** : la fonction à deux points $(30+30\sqrt2)/2304$ citée telle quelle — non rejouée.
4. **La rangée d'insertion série** et **la rangée à deux points** : prémisses ouvertes citées du ch. 10.
5. **Les huit rapports carrés de tick** (§54) : l'élément d'ordre infini existe à chacun (Pell) ; le compte des cartes par rapport n'est pas rejoué ici.

### Exercice 1 — l'exponentielle du boost au point hyperbolique rationnel

$K^2 = I$ (mesuré ci-dessus) donne $\exp(\theta K) = \cosh\theta\, I + \sinh\theta\, K$. Au point pythagoricien $(3,4,5)$ : $\cosh = 5/4$, $\sinh = 3/4$. Construire $M$, vérifier la condition **de groupe** $M^T P_L M = P_L$ — quadratique, à distinguer du résidu linéaire d'algèbre — puis lire $\tanh = \sinh/\cosh$ et le rapport $x/t$ de la première colonne.

In [12]:
// Exercice 1 : stub C.1
use std::cmp::Ordering;
use std::fmt;
use std::ops::{Add, Sub, Mul, Div, Neg};

#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: u128, mut b: u128) -> u128 {
    while b != 0 { let t = a % b; a = b; b = t; } a
}

impl Fract {
    fn new(n: i128, d: i128) -> Self {
        assert!(d != 0);
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = gcd(n.unsigned_abs(), d as u128) as i128;
        Self { num: n / g, den: d / g }
    }
    fn int(n: i128) -> Self { Self::new(n, 1) }
}

impl Add for Fract { type Output=Self; fn add(self,r:Self)->Self { Self::new(self.num*r.den+r.num*self.den,self.den*r.den) } }
impl Sub for Fract { type Output=Self; fn sub(self,r:Self)->Self { Self::new(self.num*r.den-r.num*self.den,self.den*r.den) } }
impl Mul for Fract { type Output=Self; fn mul(self,r:Self)->Self { Self::new(self.num*r.num,self.den*r.den) } }
impl Div for Fract { type Output=Self; fn div(self,r:Self)->Self { Self::new(self.num*r.den,self.den*r.num) } }
impl Neg for Fract { type Output=Self; fn neg(self)->Self { Self::new(-self.num,self.den) } }
impl PartialOrd for Fract { fn partial_cmp(&self,r:&Self)->Option<Ordering> { (self.num*r.den).partial_cmp(&(r.num*self.den)) } }
impl fmt::Display for Fract {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

let _ch = Fract::new(5, 4);
let _sh = Fract::new(3, 4);
// TODO etudiant : ch*I2 + sh*Kp, puis mul(mul(transp(M), PL), M) == PL,
// puis tanh = sh/ch et le rapport M[1][0]/M[0][0]
let _m: Option<Vec<Vec<Fract>>> = None;

println!("Exercice a completer");

Exercice a completer


### Exercice 2 — les signatures des deux moitiés

La forme de Killing mesurée est diagonale par blocs. Extraire sa restriction au sous-espace des rotations $(J_1, J_2, J_3)$ et à celui des boosts $(K_1, K_2, K_3)$ : deux signatures attendues, (0, 3) et (3, 0) — l'algèbre de directions du contre-test, et son complément.

In [13]:
// Exercice 2 : stub C.1
// TODO etudiant : signature_sym sur la sous-matrice 3x3 des J
let _sig_rot: Option<(usize, usize)> = None;
// TODO etudiant : idem sur celle des K
let _sig_boost: Option<(usize, usize)> = None;

println!("Exercice a completer");

Exercice a completer


### Exercice 3 — le lien (2,3) lu comme lien de chaîne

Le troisième contre-test : lire le lien terminal (2,3) du proton et du photon comme un lien de chaîne déplace la Gram vers $\mathrm{diag}(6, -1, -1, -1)$ — l'entrée de l'unité passe à la dimension du lien. Calculer les signes pivots et Sylvester de cette Gram : la signature bouge-t-elle ? Et son rapport unité/direction ?

In [14]:
// Exercice 3 : stub C.1
use std::cmp::Ordering;
use std::fmt;
use std::ops::{Add, Sub, Mul, Div, Neg};

#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: u128, mut b: u128) -> u128 {
    while b != 0 { let t = a % b; a = b; b = t; } a
}

impl Fract {
    fn new(n: i128, d: i128) -> Self {
        assert!(d != 0);
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = gcd(n.unsigned_abs(), d as u128) as i128;
        Self { num: n / g, den: d / g }
    }
    fn int(n: i128) -> Self { Self::new(n, 1) }
}

impl Add for Fract { type Output=Self; fn add(self,r:Self)->Self { Self::new(self.num*r.den+r.num*self.den,self.den*r.den) } }
impl Sub for Fract { type Output=Self; fn sub(self,r:Self)->Self { Self::new(self.num*r.den-r.num*self.den,self.den*r.den) } }
impl Mul for Fract { type Output=Self; fn mul(self,r:Self)->Self { Self::new(self.num*r.num,self.den*r.den) } }
impl Div for Fract { type Output=Self; fn div(self,r:Self)->Self { Self::new(self.num*r.den,self.den*r.num) } }
impl Neg for Fract { type Output=Self; fn neg(self)->Self { Self::new(-self.num,self.den) } }
impl PartialOrd for Fract { fn partial_cmp(&self,r:&Self)->Option<Ordering> { (self.num*r.den).partial_cmp(&(r.num*self.den)) } }
impl fmt::Display for Fract {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

let _g23 = vec![Fract::int(6), Fract::int(-1), Fract::int(-1), Fract::int(-1)];
// TODO etudiant : sig_pivots(G23) et sig_sylvester(G23), puis le ratio 6/(-1)
let _sig_23: Option<(usize, usize)> = None;

println!("Exercice a completer");

Exercice a completer


## Conclusion

Le groupe de Lorentz n'est pas plus posé que le groupe des rotations : les six générateurs tombent de la scission hermitienne, les constantes (80) sont exactes, la Killing sépare trois et trois, et le rapport $-4$ — l'unité contre quatre directions — tient aux deux types de lien. La signature un contre trois vit au choix de l'unité, et le transport la porte le long de chaque chaîne sans bouger (théorème 55.1) : la rotation d'unité reste euclidienne, le boost reste lorentzien, à chaque pas.

La suite (ch. 51) place le boost à l'unité : l'ouvert 56.2 et la famille de Pell à l'unité debout — grain 12.

In [15]:
// Releve final : les checks du carnet.
use std::cmp::Ordering;
use std::fmt;
use std::ops::{Add, Sub, Mul, Div, Neg};

#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct Fract { num: i128, den: i128 }

fn gcd(mut a: u128, mut b: u128) -> u128 {
    while b != 0 { let t = a % b; a = b; b = t; } a
}

impl Fract {
    fn new(n: i128, d: i128) -> Self {
        assert!(d != 0);
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = gcd(n.unsigned_abs(), d as u128) as i128;
        Self { num: n / g, den: d / g }
    }
    fn int(n: i128) -> Self { Self::new(n, 1) }
}

impl Add for Fract { type Output=Self; fn add(self,r:Self)->Self { Self::new(self.num*r.den+r.num*self.den,self.den*r.den) } }
impl Sub for Fract { type Output=Self; fn sub(self,r:Self)->Self { Self::new(self.num*r.den-r.num*self.den,self.den*r.den) } }
impl Mul for Fract { type Output=Self; fn mul(self,r:Self)->Self { Self::new(self.num*r.num,self.den*r.den) } }
impl Div for Fract { type Output=Self; fn div(self,r:Self)->Self { Self::new(self.num*r.den,self.den*r.num) } }
impl Neg for Fract { type Output=Self; fn neg(self)->Self { Self::new(-self.num,self.den) } }
impl PartialOrd for Fract { fn partial_cmp(&self,r:&Self)->Option<Ordering> { (self.num*r.den).partial_cmp(&(r.num*self.den)) } }
impl fmt::Display for Fract {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        if self.den == 1 { write!(f, "{}", self.num) } else { write!(f, "{}/{}", self.num, self.den) }
    }
}

fn mul(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let n = A.len(); let m = B[0].len(); let k = B.len();
    let mut R = vec![vec![Fract::int(0); m]; n];
    for i in 0..n { for j in 0..m {
        let mut s = Fract::int(0);
        for t in 0..k { s = s + A[i][t] * B[t][j]; }
        R[i][j] = s;
    }}
    R
}
fn add(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = A[i][j] + B[i][j]; } }
    R
}
fn sub(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = A[i][j] - B[i][j]; } }
    R
}
fn br(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    sub(&mul(A, B), &mul(B, A))
}
fn eq(A: &Vec<Vec<Fract>>, B: &Vec<Vec<Fract>>) -> bool {
    A.iter().zip(B.iter()).all(|(ra, rb)| ra.iter().zip(rb.iter()).all(|(a, b)| a == b))
}
fn transp(A: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let n = A.len();
    let mut R = vec![vec![Fract::int(0); n]; n];
    for i in 0..n { for j in 0..n { R[i][j] = A[j][i]; } }
    R
}
fn neg(A: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); A[0].len()]; A.len()];
    for i in 0..A.len() { for j in 0..A[0].len() { R[i][j] = -A[i][j]; } }
    R
}
fn zeros(n: usize) -> Vec<Vec<Fract>> {
    vec![vec![Fract::int(0); n]; n]
}
fn E(i: usize, j: usize) -> Vec<Vec<Fract>> {
    let mut M = zeros(4); M[i][j] = Fract::int(1); M
}
fn nulle(A: &Vec<Vec<Fract>>) -> bool {
    A.iter().all(|row| row.iter().all(|&v| v == Fract::int(0)))
}

let eta = vec![
    vec![Fract::int(4), Fract::int(0), Fract::int(0), Fract::int(0)],
    vec![Fract::int(0), Fract::int(-1), Fract::int(0), Fract::int(0)],
    vec![Fract::int(0), Fract::int(0), Fract::int(-1), Fract::int(0)],
    vec![Fract::int(0), Fract::int(0), Fract::int(0), Fract::int(-1)],
];

let J1 = sub(&E(3, 2), &E(2, 3));
let J2 = sub(&E(1, 3), &E(3, 1));
let J3 = sub(&E(2, 1), &E(1, 2));
fn boost(i: usize) -> Vec<Vec<Fract>> {
    let mut K = zeros(4);
    K[0][i] = Fract::new(1, 2);
    K[i][0] = Fract::int(2);
    K
}
let K1 = boost(1);
let K2 = boost(2);
let K3 = boost(3);
let base = vec![J1.clone(), J2.clone(), J3.clone(), K1.clone(), K2.clone(), K3.clone()];

// Check 1: six residus d'isometrie (eta) nuls
let residus_ok = base.iter().all(|X| {
    let R = add(&mul(&transp(X), &eta), &mul(&eta, X));
    nulle(&R)
});

// Check 2: rot-rot
let rot_rot_ok = eq(&br(&J1, &J2), &J3) && eq(&br(&J2, &J3), &J1) && eq(&br(&J3, &J1), &J2);

// Check 3: melange [Ji,Kj] = eps_ijk Kk
fn eps(i: usize, j: usize, k: usize) -> i64 {
    if i == j || j == k || i == k { return 0; }
    if (i, j, k) == (1, 2, 3) || (i, j, k) == (2, 3, 1) || (i, j, k) == (3, 1, 2) { 1 } else { -1 }
}
fn echelle(M: &Vec<Vec<Fract>>, c: Fract) -> Vec<Vec<Fract>> {
    let mut R = vec![vec![Fract::int(0); M[0].len()]; M.len()];
    for i in 0..M.len() { for j in 0..M[0].len() { R[i][j] = M[i][j] * c; } }
    R
}
let mut melange_ok = true;
for i in 1..=3 {
    for j in 1..=3 {
        let mut attendu = zeros(4);
        for k in 1..=3 {
            let e = eps(i, j, k);
            if e != 0 {
                attendu = add(&attendu, &echelle(&base[3 + k - 1], Fract::int(e as i128)));
            }
        }
        if !eq(&br(&base[i - 1], &base[3 + j - 1]), &attendu) {
            melange_ok = false;
        }
    }
}

// Check 4: boost-boost
let boost_boost_ok = eq(&br(&K1, &K2), &neg(&J3)) && eq(&br(&K2, &K3), &neg(&J1)) && eq(&br(&K3, &K1), &neg(&J2));

// Check 5: Jacobi 216/216
let total = base.len().pow(3);
let mut jacobi_reussis = 0;
for X in &base {
    for Y in &base {
        for Z in &base {
            let j = add(&add(&br(&br(X, Y), Z), &br(&br(Y, Z), X)), &br(&br(Z, X), Y));
            if nulle(&j) { jacobi_reussis += 1; }
        }
    }
}
let jacobi_ok = jacobi_reussis == total;

// Check 6: Killing diag + sig (3,3)
fn coeffs(C: &Vec<Vec<Fract>>, base: &Vec<Vec<Vec<Fract>>>) -> Option<Vec<i64>> {
    if nulle(C) { return Some(vec![0; 6]); }
    for k in 0..6 {
        if eq(C, &base[k]) { let mut v = vec![0i64; 6]; v[k] = 1; return Some(v); }
        if eq(C, &neg(&base[k])) { let mut v = vec![0i64; 6]; v[k] = -1; return Some(v); }
    }
    None
}
let mut ads: Vec<Vec<Vec<i64>>> = Vec::new();
for X in &base {
    let mut cols = Vec::new();
    for e in &base {
        match coeffs(&br(X, e), &base) {
            Some(v) => cols.push(v),
            None => cols.push(vec![0; 6]),
        }
    }
    ads.push(cols);
}
let mut kappa = vec![vec![Fract::int(0); 6]; 6];
for a in 0..6 {
    for b in 0..6 {
        let mut s = Fract::int(0);
        for j in 0..6 { for k in 0..6 {
            s = s + Fract::int(ads[a][j][k] as i128) * Fract::int(ads[b][k][j] as i128);
        }}
        kappa[a][b] = s;
    }
}
fn signature_sym(S: &Vec<Vec<Fract>>) -> (usize, usize) {
    let mut M: Vec<Vec<Fract>> = S.iter().map(|row| row.to_vec()).collect();
    let mut signes: Vec<i64> = Vec::new();
    for _ in 0..S.len() {
        let mut pr: Option<usize> = None;
        let mut best_q = Fract::int(0);
        for r in 0..M.len() {
            let a = if M[r][r] < Fract::int(0) { -M[r][r] } else { M[r][r] };
            if a > best_q { best_q = a; pr = Some(r); }
        }
        match pr {
            None => break,
            Some(p) => {
                if M[p][p] == Fract::int(0) { break; }
                signes.push(if M[p][p] > Fract::int(0) { 1 } else { -1 });
                let pivot = M[p][p];
                let garde: Vec<usize> = (0..M.len()).filter(|&i| i != p).collect();
                let mut newM = Vec::new();
                for i in &garde {
                    let mut row = Vec::new();
                    for j in &garde {
                        row.push(M[*i][*j] - M[*i][p] * M[p][*j] / pivot);
                    }
                    newM.push(row);
                }
                M = newM;
            }
        }
    }
    (signes.iter().filter(|&&s| s == 1).count(), signes.iter().filter(|&&s| s == -1).count())
}
let sig_kappa = signature_sym(&kappa);
let kappa_diag_ok = (0..6).all(|i| kappa[i][i] == (if i < 3 { Fract::int(-4) } else { Fract::int(4) }));
let killing_ok = kappa_diag_ok && sig_kappa == (3, 3);

// Check 7: (3/2)*G81 == G79
let G79 = vec![Fract::int(6), Fract::new(-3, 2), Fract::new(-3, 2), Fract::new(-3, 2)];
let G81 = vec![Fract::int(4), Fract::int(-1), Fract::int(-1), Fract::int(-1)];
let eq79_ok = (0..4).all(|i| Fract::new(3, 2) * G81[i] == G79[i]);

// Check 8: ratio -4
let ratio79 = G79[0] / G79[1];
let ratio81 = G81[0] / G81[1];
let ratio_ok = ratio79 == Fract::int(-4) && ratio81 == Fract::int(-4);

// Check 9: signatures (1,3) par pivots ET Sylvester
fn sig_pivots(diag: &Vec<Fract>) -> (usize, usize) {
    let s: Vec<i64> = diag.iter().map(|v| if *v > Fract::int(0) { 1 } else { -1 }).collect();
    (s.iter().filter(|&&x| x == 1).count(), s.iter().filter(|&&x| x == -1).count())
}
fn sig_sylvester(diag: &Vec<Fract>) -> (usize, usize) {
    let mut prods: Vec<Fract> = Vec::new();
    let mut p = Fract::int(1);
    for v in diag { p = p * *v; prods.push(p); }
    let mut suite: Vec<i64> = vec![1];
    for x in &prods { suite.push(if *x > Fract::int(0) { 1 } else { -1 }); }
    let negatifs = (0..suite.len() - 1).filter(|&i| suite[i] != suite[i + 1]).count();
    (diag.len() - negatifs, negatifs)
}
let sig79 = sig_pivots(&G79);
let sig81 = sig_pivots(&G81);
let syl79 = sig_sylvester(&G79);
let syl81 = sig_sylvester(&G81);
let sigs_ok = sig79 == (1, 3) && sig81 == (1, 3) && syl79 == (1, 3) && syl81 == (1, 3);

// Check 10: plan
let PL = vec![vec![Fract::int(4), Fract::int(0)], vec![Fract::int(0), Fract::int(-1)]];
let Dp = vec![vec![Fract::int(1), Fract::int(0)], vec![Fract::int(0), Fract::new(1, 4)]];
let Kp = vec![vec![Fract::int(0), Fract::new(1, 2)], vec![Fract::int(2), Fract::int(0)]];
let Rp = vec![vec![Fract::int(0), Fract::new(-1, 2)], vec![Fract::int(2), Fract::int(0)]];
let I2 = vec![vec![Fract::int(1), Fract::int(0)], vec![Fract::int(0), Fract::int(1)]];
let Z2 = vec![vec![Fract::int(0), Fract::int(0)], vec![Fract::int(0), Fract::int(0)]];
fn resid2(A: &Vec<Vec<Fract>>, P: &Vec<Vec<Fract>>) -> Vec<Vec<Fract>> {
    add(&mul(&transp(A), P), &mul(P, A))
}
let r_boost = resid2(&Kp, &PL);
let boost_ok = eq(&r_boost, &Z2);
let k_carre = eq(&mul(&Kp, &Kp), &I2);
let r_carre = eq(&mul(&Rp, &Rp), &neg(&I2));
let plan_ok = boost_ok && k_carre && r_carre;

// Check 11: R euclidien et non-Gram
let r_eucl = resid2(&Rp, &Dp);
let r_eucl_ok = eq(&r_eucl, &Z2);
let r_gram = resid2(&Rp, &PL);
let r_gram_ok = eq(&r_gram, &vec![vec![Fract::int(0), Fract::int(-4)], vec![Fract::int(-4), Fract::int(0)]]);
let r_eucl_gram_ok = r_eucl_ok && r_gram_ok;

// Check 12: rotations anti-hermitiennes
#[derive(Clone, Copy, Debug, PartialEq, Eq)]
struct C { re: Fract, im: Fract }
fn cscale(z: C, n: Fract) -> C {
    C { re: z.re * n, im: z.im * n }
}
let C0 = C { re: Fract::int(0), im: Fract::int(0) };
let CI = C { re: Fract::int(0), im: Fract::int(1) };
let G1 = vec![
    vec![C0, cscale(CI, Fract::new(-1, 2))],
    vec![cscale(CI, Fract::new(-1, 2)), C0],
];
let G2 = vec![
    vec![C0, C { re: Fract::new(1, 2), im: Fract::int(0) }],
    vec![C { re: Fract::new(-1, 2), im: Fract::int(0) }, C0],
];
let G3 = vec![
    vec![cscale(CI, Fract::new(-1, 2)), C0],
    vec![C0, cscale(CI, Fract::new(1, 2))],
];
let G2_flip: Vec<Vec<C>> = G2.iter().map(|row| row.iter().map(|z| C { re: -z.re, im: -z.im }).collect()).collect();
let T = vec![G1, G2_flip, G3];
fn dagger(M: &Vec<Vec<C>>) -> Vec<Vec<C>> {
    let mut R = vec![vec![C { re: Fract::int(0), im: Fract::int(0) }; 2]; 2];
    for i in 0..2 { for j in 0..2 {
        R[i][j] = C { re: M[j][i].re, im: -M[j][i].im };
    }}
    R
}
fn meq2(A: &Vec<Vec<C>>, B: &Vec<Vec<C>>) -> bool {
    for i in 0..2 { for j in 0..2 { if A[i][j] != B[i][j] { return false; } } }
    true
}
fn negc(M: &Vec<Vec<C>>) -> Vec<Vec<C>> {
    M.iter().map(|row| row.iter().map(|z| C { re: -z.re, im: -z.im }).collect()).collect()
}
let anti: Vec<bool> = T.iter().map(|t| meq2(&dagger(t), &negc(t))).collect();
let anti_ok = anti.iter().all(|&b| b);

// Check 13: cone nul
let mut cone_ok = true;
for v in [(1i64, 2i64), (1i64, -2i64)] {
    let q = PL[0][0] * Fract::int(v.0 as i128) * Fract::int(v.0 as i128) + PL[1][1] * Fract::int(v.1 as i128) * Fract::int(v.1 as i128);
    if q != Fract::int(0) { cone_ok = false; }
}

// Check 14: Pell
let pell: i64 = 3*3 - 2*2*2;
let pell_ok = pell == 1;

// Assemblage des 14 checks
let checks: Vec<(&str, bool)> = vec![
    ("six residus d'isometrie (eta) nuls", residus_ok),
    ("rot-rot [J1,J2]=J3 (x3)", rot_rot_ok),
    ("melange [Ji,Kj]=eps_ijk Kk (x9)", melange_ok),
    ("boost-boost [K1,K2]=-J3 (x3)", boost_boost_ok),
    ("Jacobi 216/216 triples", jacobi_ok),
    ("Killing diag(-4,-4,-4,4,4,4), sig (3,3)", killing_ok),
    ("(3/2)*diag(4,..) = eq.79", eq79_ok),
    ("ratio -4 aux deux liens", ratio_ok),
    ("signatures (1,3) par pivots ET Sylvester", sigs_ok),
    ("plan : residu boost nul, K^2=I, R^2=-I", plan_ok),
    ("R euclidien (residu nul) et non-Gram ([[0,-4],[-4,0]])", r_eucl_gram_ok),
    ("rotations anti-hermitiennes (x3)", anti_ok),
    ("cone nul (1,2) et (1,-2)", cone_ok),
    ("Pell 3^2 - 2*2^2 = 1", pell_ok),
];

fn py_bool(b: bool) -> String {
    if b { "True".to_string() } else { "False".to_string() }
}

let ok = checks.iter().filter(|c| c.1).count();
for &(k, v) in &checks {
    println!("{} -> {}", k, py_bool(v));
}
println!("Total : {} / {}", ok, checks.len());

six residus d'isometrie (eta) nuls -> True


rot-rot [J1,J2]=J3 (x3) -> True


melange [Ji,Kj]=eps_ijk Kk (x9) -> True


boost-boost [K1,K2]=-J3 (x3) -> True


Jacobi 216/216 triples -> True


Killing diag(-4,-4,-4,4,4,4), sig (3,3) -> True


... (5 outputs left out)

rotations anti-hermitiennes (x3) -> True


cone nul (1,2) et (1,-2) -> True


Pell 3^2 - 2*2^2 = 1 -> True


Total : 14 / 14


## À retenir

1. **Le groupe de Lorentz se dérive, comme les rotations** : la même scission hermitienne de la
   trace donne six générateurs — trois $J$ anti-hermitiens (les rotations), trois $K$ hermitiens
   (les boosts) — et les six résidus d'isométrie sont nuls.
2. **Les constantes de structure (80) sont exactes** : $[J_1,J_2] = J_3$, les neuf mélanges
   $[J_i, K_j] = \epsilon_{ijk} K_k$, $[K_1, K_2] = -J_3$ — et l'identité de Jacobi tient sur les
   $216$ triples de la base.
3. **La forme de Killing sépare trois et trois** : $\kappa = \mathrm{diag}(-4,-4,-4,\,4,4,4)$,
   signature $(3,3)$ par élimination symétrique exacte — les rotations portent $(0,3)$, les boosts
   $(3,0)$, et c'est la lecture du contre-test du source.
4. **Un contre trois aux deux liens** : $G_{79} = \tfrac32\,\mathrm{diag}(4,-1,-1,-1)$ et
   $G_{81} = \mathrm{diag}(4,-1,-1,-1)$, tous deux de signature $(1,3)$ par pivots **et** par
   Sylvester, avec le même rapport $-4$ — l'unité contre les quatre directions.
5. **La signature vit au choix de l'unité, et le transport la porte** : la rotation d'unité reste
   euclidienne ($R^2 = -I$, résidu nul, et sa « Gram » $[[0,-4],[-4,0]]$ n'en est pas une), le
   boost reste lorentzien ($K^2 = I$), le cône nul porte $(1,2)$ et $(1,-2)$, et
   $3^2 - 2 \cdot 2^2 = 1$ est la famille de Pell à l'unité debout.

## Vérifiez votre compréhension

1. Pourquoi les rotations sont-elles anti-hermitiennes et les boosts hermitiens, et que mesure ce
   partage ?
2. Que vaut la forme de Killing sur cette algèbre, et comment s'y lit la signature $(3,3)$ ?
3. Les deux Grams de l'éq. 79 et de l'éq. 81 n'ont pas les mêmes coefficients : pourquoi donnent-ils
   pourtant la même signature ?
4. Qu'est-ce qui distingue le plan de la rotation d'unité de celui du boost, et où le théorème 55.1
   intervient-il ?

<details>
<summary>Réponses</summary>

1. Elles tombent de la scission hermitienne : $J = i\sigma/2$ est anti-hermitien, $K = \sigma/2$
   est hermitien. Le partage est exactement ce que la forme de Killing mesure — $\kappa(J,J) = -4$,
   $\kappa(K,K) = +4$, tous les croisés nuls.
2. $\kappa = \mathrm{diag}(-4,-4,-4,\,4,4,4)$ : trois négatifs (les rotations) et trois positifs
   (les boosts), d'où la signature $(3,3)$ — et, séparément, $(0,3)$ sur les rotations et $(3,0)$
   sur les boosts.
3. Ils ne diffèrent que par le facteur scalaire $3/2$ : $G_{79} = \tfrac32 G_{81}$. Un facteur
   positif ne change ni le signe ni le nombre des pivots, donc les deux formes ont la même
   signature $(1,3)$ — et le rapport des deux vaut $-4$ dans les deux cas.
4. La rotation d'unité vérifie $R^2 = -I$ — elle est euclidienne, de résidu nul, et sa forme
   associée $[[0,-4],[-4,0]]$ n'est pas une Gram (non définie) ; le boost vérifie $K^2 = I$ et son
   cône nul porte exactement $(1,2)$ et $(1,-2)$. Le théorème 55.1 est le transport : la signature
   un-contre-trois ne bouge pas le long de la chaîne.

</details>

## Pour aller plus loin

*Chaînes-12 — les boosts et l'unité une* est le
carnet que la conclusion annonce : le boost à l'unité, l'ouvert 56.2 et la famille de Pell à l'unité
debout.

En amont, [Chaînes-10 — le groupe des rotations](Chaines-10-Groupe-Rotations-Automorphismes-Directions-Python.ipynb)
est le carnet dont la conclusion annonçait celui-ci (« the next chapter joins them to the order
coordinate in the Lorentz group »), et [Chaînes-09 — la coordonnée d'ordre](Chaines-09-Coordonnee-Ordre-Chaine-Serie-Python.ipynb)
fournit la coordonnée que le théorème 55.1 transporte le long de la chaîne.

Hors de la série, [Jauge-01 — le théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb)
reste le contraste de méthode : là où une conservation se dérive d'une symétrie continue, les six
générateurs de ce carnet sont **dérivés d'une scission**, et c'est la forme de Killing — non un
postulat — qui sépare les trois rotations des trois boosts.

Côté géométrie : le modèle de ce carnet, $\mathbb{R}^4$ muni de $\eta = \mathrm{diag}(4, -1, -1, -1)$,
est une variété pseudo-riemannienne — $\eta$ est bilinéaire, non dégénérée, non définie positive, de
signature $(1, 3)$ — et son groupe d'isométries linéaires est isomorphe au groupe de Lorentz. Senia
Sheydvasser, *An Overview of Differential Geometry*, The Deranged Mathematician, 30/09/2026 —
<https://derangedmathematician.substack.com/p/an-overview-of-differential-geometry> — y présente l'espace
de Minkowski comme la variété pseudo-riemannienne la plus simple qui soit d'un grand intérêt : la forme
n'y définit pas une distance, mais l'intervalle d'espace-temps.

> Carnet issu de l'EPIC #26534.